# STEM vs Non-STEM Distribution Analysis

Visualizing the split between STEM and Non-STEM programs among GDG applicants.

In [ ]:
# Import required libraries
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# Read SQL query from stem.sql and connect to database
db_path = '../../db/applications.db'
conn = sqlite3.connect(db_path)

# Read query from stem.sql file
with open('stem.sql', 'r') as f:
    query = f.read()

# Fetch STEM distribution data
stem_data = pd.read_sql_query(query, conn)

print(f"Total categories: {len(stem_data)}")
print(f"Total applicants: {stem_data['total_count'].sum()}")
print("\nSTEM Distribution with Acceptance/Rejection:")
print(stem_data)
conn.close()

In [ ]:
# Create STEM distribution visualization with acceptance rates
fig, ax = plt.subplots(figsize=(12, 6))

# Calculate acceptance rates
stem_data['acceptance_rate'] = (stem_data['acceptence_count'] / stem_data['total_count'] * 100).round(2)

# Create grouped bar chart for accepted vs rejected
x = range(len(stem_data))
width = 0.35

bars1 = ax.bar([i - width/2 for i in x], stem_data['acceptence_count'], width,
               label='Accepted', color='#34A853', edgecolor='black', alpha=0.8, linewidth=1.2)
bars2 = ax.bar([i + width/2 for i in x], stem_data['rejection_count'], width,
               label='Rejected', color='#EA4335', edgecolor='black', alpha=0.8, linewidth=1.2)

# Customize appearance
ax.set_title('STEM vs Non-STEM: Applications & Acceptance Rates', fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel('Program Type', fontsize=12, fontweight='bold')
ax.set_ylabel('Number of Applicants', fontsize=12, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(stem_data['program_type'])
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y', linestyle='--')
ax.set_axisbelow(True)

# Add value labels
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{int(height)}', ha='center', va='bottom', fontsize=9, fontweight='bold')

# Add acceptance rate labels on top
for i, rate in enumerate(stem_data['acceptance_rate']):
    ax.text(i, stem_data['total_count'].iloc[i] + 5,
            f'{rate:.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold', color='#1a73e8')

# Statistics box
total = stem_data['total_count'].sum()
total_accepted = stem_data['acceptence_count'].sum()
overall_rate = (total_accepted / total * 100).round(2)
stats_text = f'Total: {total} | Accepted: {total_accepted} | Overall Rate: {overall_rate}%'
ax.text(0.5, 1, stats_text, transform=fig.transFigure,
        ha='center', va='top', fontsize=11, bbox=dict(boxstyle='round', facecolor='#fff3cd', alpha=0.8))

plt.tight_layout()
plt.show()

In [ ]:
# Read program types query and fetch fresh data
import sqlite3
import pandas as pd

# Connect, read query file, fetch, and close
prog_db_path = '../../db/applications.db'
conn = sqlite3.connect(prog_db_path)
with open('programs.sql', 'r') as f:
    programs_query = f.read()
programs_df = pd.read_sql_query(programs_query, conn)
conn.close()

# Preview
print('Programs breakdown (refreshed):')
print(programs_df)

In [ ]:
# Visualize program type distribution
import matplotlib.pyplot as plt

# Ensure expected columns exist
if {'program_type', 'program_count'}.issubset(programs_df.columns):
    # Sort for nicer display
    programs_df = programs_df.sort_values('program_count', ascending=False)

    fig, ax = plt.subplots(figsize=(12, 6))
    colors = ['#4285F4', '#34A853', '#FBBC04', '#EA4335', '#AB47BC', '#00BCD4', '#FF6F00']
    bars = ax.bar(programs_df['program_type'], programs_df['program_count'],
                  color=colors[:len(programs_df)], edgecolor='black', alpha=0.85, linewidth=1.2)

    ax.set_title('Applicants by Program Type', fontsize=16, fontweight='bold', pad=20)
    ax.set_xlabel('Program Type', fontsize=12, fontweight='bold')
    ax.set_ylabel('Number of Applicants', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y', linestyle='--')
    ax.set_axisbelow(True)
    plt.xticks(rotation=45, ha='right')

    # Labels
    total = programs_df['program_count'].sum()
    for bar, count in zip(bars, programs_df['program_count']):
        ax.text(bar.get_x() + bar.get_width()/2., count,
                f'{int(count)}', ha='center', va='bottom', fontsize=10, fontweight='bold')
        pct = (count / total) * 100
        ax.text(bar.get_x() + bar.get_width()/2., count/2,
                f'{pct:.1f}%', ha='center', va='center', fontsize=10, fontweight='bold', color='white')

    stats_text = f'Total Applicants: {total}'
    ax.text(0.5, 0.98, stats_text, transform=fig.transFigure,
            ha='center', va='top', fontsize=11,
            bbox=dict(boxstyle='round', facecolor='#fff3cd', alpha=0.8))

    plt.tight_layout()
    plt.show()
else:
    print('Expected columns not found in programs_df:', programs_df.columns.tolist())
